<a href="https://colab.research.google.com/github/kunphat510214-netizen/Final-Final/blob/step-5-6/Coffee_Shop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ขั้นตอนที่ 2 -- ออกแบบ Class

*   List item
*   List item


ออกแบบ Class — สร้างอย่างน้อย 3 class ที่จำลอง "สิ่งของ" หลักในธุรกิจ แต่ละ class ต้องมี attribute และ method ที่สมเหตุสมผล

In [4]:
# ---------------------------------------
# 1. 📦 การนำเข้าไลบรารี (Import Libraries)
# ---------------------------------------
import random
import sqlite3
import pandas as pd

# 📊 ตรวจสอบการนำเข้า matplotlib สำหรับวาดกราฟ (ป้องกันโปรแกรมค้างหากไม่ได้ติดตั้ง)
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

# 🖥️ ตรวจสอบการนำเข้า display สำหรับแสดงผลใน Jupyter Notebook/Colab
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

# --------------------------------------
# 2. 👤 คลาสสำหรับจัดการข้อมูลสมาชิก (Member)
# --------------------------------------
class Member:
    def __init__(self, member_id, name, points=0):
        # 💳 กำหนดค่าเริ่มต้นให้กับสมาชิก: รหัสสมาชิก, ชื่อ, และแต้มสะสมเริ่มต้น
        self.member_id = member_id
        self.name = name
        self.points = points # ⭐️ แต้มสะสมแรกรับ

    def add_points(self, cups=1):
        # 🪙 ฟังก์ชันสำหรับสะสมแต้มตามจำนวนแก้วที่สั่งซื้อ
        self.points += cups
        return self.points # ⭐️ คืนค่าแต้มสะสมล่าสุด

# ----------------------------------------------
# 3. ☕️ คลาสสำหรับจัดการออเดอร์เครื่องดื่ม (DrinkOrder)
# ----------------------------------------------
class DrinkOrder:
    # --- 💸 กำหนดราคามาตรฐานและตัวเลือกต่างๆ (Class Attributes) ---
    BASE_PRICES = {
        "อเมริกาโน่": 45, "ลาเต้": 50, "คาปูชิโน่": 50,
        "เอสเพรสโซ่": 50, "มอคค่า": 50, "ชาเขียว": 45,
        "ชาไทย": 45, "มัทฉะลาเต้": 50, "ช็อกโกแลต": 50,
        "นมสด": 40, "สตรอว์เบอร์รี่โซดา": 40,
        "บลูฮาวายโซดา": 40, "ยูซุโซดา": 40,
    }
    TYPE_PRICES = {"ร้อน": 0, "เย็น": 5, "ปั่น": 10}
    SIZE_PRICES = {"S": 0, "M": 10, "L": 20}
    TOPPING_PRICES = {"ไม่มี": 0, "ไข่มุก": 5, "วิปครีม": 15, "ครีมชีส": 20, "เจลลี่": 5}
    PAYMENT_METHODS = ("เงินสด", "QR Code", "บัตร")

    def __init__(self, order_id, customer_name, menu_name, drink_type, size,
                 topping, sweetness_level, receive_type, member_id=None):
        # 📝 จัดเก็บข้อมูลและรายละเอียดออเดอร์
        self.order_id = order_id
        self.queue_no = f"Q{order_id:03d}"   # 🏷️ สร้างหมายเลขคิว เช่น Q001
        self.customer_name = customer_name
        self.menu_name = menu_name
        self.drink_type = drink_type
        self.size = size
        self.topping = topping
        self.sweetness_level = sweetness_level
        self.receive_type = receive_type
        self.member_id = member_id

        # ⏳ สถานะเริ่มต้นของการชำระเงินและการดำเนินงาน
        self.payment_method = None
        self.payment_status = "ยังไม่ชำระ"
        self.status = "รอชำระเงิน"
        self.wait_minutes = 0

    def calculate_subtotal(self):
        # 🧮 คำนวณราคารวมขั้นต้น (ราคาฐาน + ประเภทเครื่องดื่ม + ขนาด + ท็อปปิ้ง)
        return (self.BASE_PRICES[self.menu_name]
                + self.TYPE_PRICES[self.drink_type]
                + self.SIZE_PRICES[self.size]
                + self.TOPPING_PRICES[self.topping])

    def calculate_price(self):
        # 🏷️ คำนวณราคาสุทธิ (หากเป็นสมาชิกจะได้รับส่วนลด 5%)
        # 🎁 สมาชิกได้รับส่วนลด 5%
        discount_rate = 0.05 if self.member_id else 0
        return round(self.calculate_subtotal() * (1 - discount_rate), 2)

    def confirm_payment(self, payment_method):
        # 💵 ยืนยันการชำระเงินและอัปเดตสถานะออเดอร์
        if payment_method not in self.PAYMENT_METHODS:
            raise ValueError("ช่องทางชำระเงินไม่ถูกต้อง")
        self.payment_method = payment_method
        self.payment_status = "ชำระแล้ว"
        self.status = "รอดำเนินการ"

    def receipt_text(self):
        # 🧾 ข้อความรูปแบบใบเสร็จรับเงิน
        return (f"ใบเสร็จ R{self.order_id:05d} | {self.queue_no} | "
                f"{self.menu_name} {self.drink_type} {self.size} | "
                f"{self.calculate_price():,.2f} บาท | {self.payment_method}")

    def sticker_text(self):
        # 🏷️ ข้อความรูปแบบสติกเกอร์สำหรับติดแก้วน้ำ 🥤
        return (f"{self.queue_no} {self.menu_name}/{self.drink_type}/{self.size} "
                f"หวาน {self.sweetness_level} ท็อปปิ้ง {self.topping} ({self.receive_type})")

    def to_record(self):
        # 🗂️ แปลงข้อมูลออเดอร์เป็น Dictionary เพื่อสะดวกในการบันทึกลงฐานข้อมูลหรือนำไปสร้าง DataFrame
        return {
            "order_id": self.order_id,
            "queue_no": self.queue_no,
            "customer_name": self.customer_name,
            "member_id": self.member_id,
            "menu_name": self.menu_name,
            "drink_type": self.drink_type,
            "size": self.size,
            "topping": self.topping,
            "sweetness_level": self.sweetness_level,
            "receive_type": self.receive_type,
            "payment_method": self.payment_method,
            "subtotal": self.calculate_subtotal(),
            "price": self.calculate_price(),
            "wait_minutes": self.wait_minutes,
            "status": self.status,
        }

# ----------------------------------------------
# 4. 👩🏻‍🍳 คลาสสำหรับจัดการระบบคิวและครัว (QueueSystem)
# ----------------------------------------------
class QueueSystem:
    # 📋 รายการออเดอร์ที่กำลังรอดำเนินการ และประวัติเหตุการณ์ (Log)
    def __init__(self):
        self.active_orders = []
        self.events = []
    # 📌 ฟังก์ชันภายในสำหรับบันทึกประวัติการเปลี่ยนแปลงสถานะของออเดอร์
    def _log(self, order, event):
        self.events.append({
            "event_id": len(self.events) + 1,
            "order_id": order.order_id,
            "event": event,
            "status": order.status,
        })

    def send_to_kds(self, order):
        # 📲 ส่งออเดอร์เข้าระบบห้องครัว (Kitchen Display System) หลังจากชำระเงินแล้ว
        if order.payment_status != "ชำระแล้ว":
            raise ValueError("ต้องยืนยันการชำระเงินก่อนส่งเข้า KDS")
        self.active_orders.append(order)
        order.status = "รอดำเนินการ"
        self._log(order, "ส่งเข้า KDS")

    def start_preparing(self, order):
        # 👩🏻‍🍳 บาริสต้าเริ่มชง/ทำตามออเดอร์
        order.status = "กำลังทำ"
        self._log(order, "บาริสต้ารับออเดอร์")

    def call_queue(self, order, announce=False):
        # 🔔 เรียกคิวเมื่อทำเครื่องดื่มเสร็จเรียบร้อย
        order.status = "พร้อมรับ"
        self._log(order, "เรียกคิว")
        message = f"🔔 คิว {order.queue_no} พร้อมรับที่เคาน์เตอร์"
        if announce:
            print(message, "(เสียงเรียก)")
        return message

    def complete_order(self, order):
        # 🎉 ส่งมอบเครื่องดื่มให้ลูกค้าเสร็จสิ้น และลบออกจากรายการคิวที่รอดำเนินการ
        order.status = "เสร็จสิ้น"
        self._log(order, "ส่งมอบสำเร็จ")
        self.active_orders = [o for o in self.active_orders if o.order_id != order.order_id]

In [5]:
# ขั้นที่ 3: เขียน functions ช่วยทำงาน
import random

def generate_customer_name():
    first_names = ["เจสซี่", "ลีโอ", "มายา", "อเล็กซ์", "คลาร่า", "ลูคัส", "นิโคล"]
    last_names = ["รักเรียน", "ใจดี", "สายทอง", "รุ่งเรือง", "มั่นคง", "วงศ์สว่าง", "เจริญพร"]
    return f"{random.choice(first_names)} {random.choice(last_names)}"


In [6]:
# ฟังก์ชันเลือกค่าระดับความหวานแบบสุ่ม
def random_sweetness_level(levels=("0%", "25%", "50%", "75%", "100%")):
    # เลือกค่าระดับความหวานแบบสุ่มจากรายการที่กำหนดใน 'levels'
    # โดย 'levels' มีค่าเริ่มต้นเป็น ('0%', '25%', '50%', '75%', '100%') หากไม่ได้ระบุ
    return random.choice(levels)

In [7]:
# ฟังก์ชันสำหรับตรวจสอบสถานะคำสั่งซื้อ
def check_order_status(order):
    return order.status

In [8]:
# ฟังก์ชันสำหรับสร้างคำสั่งซื้อแบบสุ่ม
def build_random_order(order_id, member=None):
    return DrinkOrder(
        order_id=order_id,
        customer_name=member.name if member else generate_customer_name(),
        menu_name=random.choice(list(DrinkOrder.BASE_PRICES)),
        drink_type=random.choice(list(DrinkOrder.TYPE_PRICES)),
        size=random.choice(list(DrinkOrder.SIZE_PRICES)),
        topping=random.choice(list(DrinkOrder.TOPPING_PRICES)),
        sweetness_level=random_sweetness_level(),
        receive_type=random.choice(["ทานที่ร้าน", "กลับบ้าน"]),
        member_id=member.member_id if member else None,
    )

# ส่วนสาธิตการทำงานของระบบ (Demo)
# 1. สร้างสมาชิกตัวอย่าง
demo_member = Member("M001", "มายา ใจดี", points=4)
# 2. สร้างคำสั่งซื้อตัวอย่างสำหรับสมาชิก
demo_order = DrinkOrder(1, demo_member.name, "ลาเต้", "เย็น", "M", "วิปครีม",
                        "50%", "กลับบ้าน", demo_member.member_id)
# 3. สร้างระบบคิว
demo_queue = QueueSystem()

# แสดงข้อมูลการรับลูกค้าและตัวเลือกของเครื่องดื่ม
print("1-3) รับลูกค้าและตัวเลือก:", demo_order.menu_name, demo_order.size,
      demo_order.topping, demo_order.sweetness_level, demo_order.receive_type,
      "สมาชิก" if demo_order.member_id else "ไม่เป็นสมาชิก")
# 4. แสดงสรุปราคาเครื่องดื่ม
print("4) สรุปราคา:", demo_order.calculate_price(), "บาท")
# ยืนยันการชำระเงินด้วย QR Code
demo_order.confirm_payment("QR Code")
# สมาชิกได้รับคะแนนเพิ่ม
demo_member.add_points()
# 5. แสดงสถานะการชำระเงินและช่องทางการชำระเงิน
print("5) ชำระเงิน:", demo_order.payment_status, "ด้วย", demo_order.payment_method)
# 6. แสดงใบเสร็จ
print("6)", demo_order.receipt_text())
# แสดงข้อความบนสติกเกอร์
print("   Sticker:", demo_order.sticker_text())
# 7. ส่งคำสั่งซื้อเข้า KDS (Kitchen Display System)
demo_queue.send_to_kds(demo_order)
print("7) KDS:", demo_order.status)
# บาริสต้าเริ่มเตรียมเครื่องดื่ม
demo_queue.start_preparing(demo_order)
# เรียกคิวเมื่อเครื่องดื่มพร้อมรับ
demo_queue.call_queue(demo_order, announce=True)
# 8-9. แสดงหน้าจอคิวและสถานะคำสั่งซื้อ
print("8-9) หน้าจอแสดง:", demo_order.queue_no, demo_order.status)
# ทำเครื่องดื่มเสร็จสมบูรณ์
demo_queue.complete_order(demo_order)
# 10. แสดงสถานะสุดท้ายของคำสั่งซื้อและจำนวนคิวที่ยังคงเหลืออยู่
print("10) สถานะ:", demo_order.status, "| คิวคงเหลือ:", len(demo_queue.active_orders))

1-3) รับลูกค้าและตัวเลือก: ลาเต้ M วิปครีม 50% กลับบ้าน สมาชิก
4) สรุปราคา: 76.0 บาท
5) ชำระเงิน: ชำระแล้ว ด้วย QR Code
6) ใบเสร็จ R00001 | Q001 | ลาเต้ เย็น M | 76.00 บาท | QR Code
   Sticker: Q001 ลาเต้/เย็น/M หวาน 50% ท็อปปิ้ง วิปครีม (กลับบ้าน)
7) KDS: รอดำเนินการ
🔔 คิว Q001 พร้อมรับที่เคาน์เตอร์ (เสียงเรียก)
8-9) หน้าจอแสดง: Q001 พร้อมรับ
10) สถานะ: เสร็จสิ้น | คิวคงเหลือ: 0


#ขั้นที่ 4 Loop

In [9]:
# ขั้นที่ 4: จำลองธุรกรรมทีละรายการด้วย loop อย่างน้อย 300 รายการ
random.seed(612104)
members = {
    f"M{i:03d}": Member(f"M{i:03d}", generate_customer_name(), random.randint(0, 9))
    for i in range(1, 61)
}

#สร้างระบบจัดการคิว และสร้างลิสต์สำหรับเก็บข้อมูลออเดอร์ทั้งหมด
queue_system = QueueSystem()
orders = []

#วนลูปเพื่อจำลองธุรกรรมจำนวน 300 รายการ
for i in range(1, 301):

  #สุ่มเลือกสมาชิก โดยมีโอกาส 40% ที่ออเดอร์จะเป็นของสมาชิก
    member = random.choice(list(members.values())) if random.random() < 0.40 else None

    #สร้างออเดอร์แบบสุ่ม พร้อมระบุหมายเลขออเดอร์และสมาชิก
    order = build_random_order(i, member)

    #ยืนยันการชำระเงินด้วยวิธีการชำระเงินแบบสุ่ม
    order.confirm_payment(random.choice(DrinkOrder.PAYMENT_METHODS))

    #หากเป็นสมาชิก ให้เพิ่มคะแนนสะสม
    if member:
        member.add_points()

   #ส่งออเดอร์เข้าสู่ระบบครัว (KDS)
    queue_system.send_to_kds(order)

    #เริ่มกระบวนการเตรียมเครื่องดื่ม
    queue_system.start_preparing(order)

    #สุ่มเวลารอรับเครื่องดื่มระหว่าง 2-18 นาที
    order.wait_minutes = random.randint(2, 18)

    #เรียกลูกค้าตามคิว
    queue_system.call_queue(order)

    #เปลี่ยนสถานะออเดอร์เป็นเสร็จสมบูรณ์
    queue_system.complete_order(order)

    #เก็บออเดอร์ที่ดำเนินการสร้างแล้วไว้ในลิสต์
    orders.append(order)

#ตรวจสอบว่ามีออเดอร์ครบ 300 รายการ
assert len(orders) == 300

#ตรวจสอบว่าไม่มีออเดอร์ค้างอยู่ในระบบ
assert len(queue_system.active_orders) == 0

#แสดงผลการจำลองธุรกรรม
print(f"จำลองครบ {len(orders)} ออเดอร์ และส่งมอบทุกคิวเรียบร้อย")

จำลองครบ 300 ออเดอร์ และส่งมอบทุกคิวเรียบร้อย


#ขั้นที่ 5-6

In [10]:
# ขั้นที่ 5-6: บันทึก CSV และฐานข้อมูล SQLite 3 ตาราง
records = [order.to_record() for order in orders]
df = pd.DataFrame(records)
df.to_csv("coffee_orders.csv", index=False, encoding="utf-8-sig")

members_df = pd.DataFrame([
    {"member_id": member.member_id, "member_name": member.name, "points": member.points}
    for member in members.values()
])
events_df = pd.DataFrame(queue_system.events)

conn = sqlite3.connect("coffee_shop.db")
df.to_sql("orders", conn, if_exists="replace", index=False)
members_df.to_sql("members", conn, if_exists="replace", index=False)
events_df.to_sql("order_events", conn, if_exists="replace", index=False)
print("บันทึก coffee_orders.csv และ coffee_shop.db สำเร็จ")
display(df.head())

บันทึก coffee_orders.csv และ coffee_shop.db สำเร็จ


,order_id,queue_no,customer_name,member_id,menu_name,drink_type,size,topping,sweetness_level,receive_type,payment_method,subtotal,price,wait_minutes,status
0,1,Q001,ลีโอ สายทอง,None,อเมริกาโน่,เย็น,M,ไข่มุก,0%,ทานที่ร้าน,บัตร,65,65.0,15,เสร็จสิ้น
1,2,Q002,ลีโอ รักเรียน,M026,ลาเต้,ร้อน,L,ครีมชีส,100%,กลับบ้าน,บัตร,90,85.5,12,เสร็จสิ้น
2,3,Q003,นิโคล มั่นคง,M018,คาปูชิโน่,ปั่น,S,ไม่มี,50%,ทานที่ร้าน,บัตร,60,57.0,3,เสร็จสิ้น
3,4,Q004,ลูคัส มั่นคง,M047,มัทฉะลาเต้,เย็น,M,วิปครีม,0%,กลับบ้าน,เงินสด,80,76.0,2,เสร็จสิ้น
4,5,Q005,อเล็กซ์ รักเรียน,None,มัทฉะลาเต้,ปั่น,S,ไม่มี,75%,ทานที่ร้าน,บัตร,60,60.0,11,เสร็จสิ้น
